# PubMed corpus collection and LLM retrieval refinement

This notebook collects recall-oriented PubMed candidates and uses a
pathogen-specific LLM refinement pass to remove abstracts that are not
actually about the target pathogen. It writes a filtered
`refined_corpus_articles.parquet` for the category notebook. All outputs are
local, resumable Parquet checkpoints under `outputs/pubmed_screening/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'assets').exists():
    raise FileNotFoundError('Could not locate the repository assets directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graphicalizer import (
    OpenAIChatCompleter,
    PathogenSpec,
    PubMedClient,
    build_pathogen_search_query,
    collect_pubmed_corpus,
    load_refinement_run,
    load_search_bundle,
    refine_pubmed_corpus,
)

## Configure pathogens, YAML search bundles, and checkpoints

In [ ]:
from graphicalizer.notebook_config import (
    configured_env,
    debug_pathogens,
    limit_debug_abstracts,
    load_notebook_config,
    resolve_config_path,
)

CONFIG = load_notebook_config(PROJECT_ROOT)
COMMON = CONFIG['common']
SETTINGS = CONFIG['notebook_02_pubmed_abstract_download']
DEBUG_MODE = COMMON['debug_mode']
DEBUG_PATHOGENS = debug_pathogens(COMMON)
DEBUG_ABSTRACTS_PER_PATHOGEN = COMMON['debug_abstracts_per_pathogen']

PUBMED_EMAIL = COMMON['pubmed_email']
PATHOGENS = DEBUG_PATHOGENS if DEBUG_MODE else COMMON['pathogens']
SEARCH_BUNDLE_DIR = resolve_config_path(PROJECT_ROOT, SETTINGS['search_bundle_dir'])
SEARCH_BUNDLES = {
    bundle_spec['search_type']: load_search_bundle(
        SEARCH_BUNDLE_DIR / bundle_spec['filename'],
        expected_search_type=bundle_spec['search_type'],
    )
    for bundle_spec in SETTINGS['search_bundles']
}

PUBMED_MAX_RESULTS_PER_QUERY = (
    DEBUG_ABSTRACTS_PER_PATHOGEN
    if DEBUG_MODE
    else SETTINGS['pubmed_max_results_per_query']
)
PUBMED_SEARCH_PAGE_SIZE = COMMON['pubmed_search_page_size']
PUBMED_FETCH_BATCH_SIZE = COMMON['pubmed_fetch_batch_size']
RESUME = COMMON['resume']
OUTPUT_DIR = resolve_config_path(
    PROJECT_ROOT,
    Path(COMMON['output_root'])
    / COMMON['corpus_subdir']
    / ('debug' if DEBUG_MODE else ''),
)

OPENAI_MODEL = configured_env(COMMON, 'openai_model_env', COMMON['openai_model'])
OPENAI_API_KEY = configured_env(COMMON, 'openai_api_key_env')
LLM_MAX_TOKENS = SETTINGS['openai_max_tokens']
LLM_RETRIES = COMMON['llm_retries']
LLM_MAX_CALLS = COMMON['llm_max_calls']
LLM_SAVE_EVERY = COMMON['llm_save_every']
RETRY_FAILED_LLM_ROWS = COMMON['retry_failed_llm_rows']

pubmed = PubMedClient(
    email=PUBMED_EMAIL,
    api_key=configured_env(COMMON, 'pubmed_api_key_env'),
    tool=COMMON['pubmed_tool'],
    retries=COMMON['pubmed_retries'],
)


## Preview the generated queries

In [ ]:
for pathogen, aliases in PATHOGENS.items():
    spec = PathogenSpec(pathogen, tuple(aliases))
    print(f'--- {pathogen} / animal ---')
    print(build_pathogen_search_query(spec, 'animal', search_bundles=SEARCH_BUNDLES))
    print(f'--- {pathogen} / zoonosis ---')
    print(build_pathogen_search_query(spec, 'zoonosis', search_bundles=SEARCH_BUNDLES))

## Collect and resume the deduplicated corpus

In [ ]:
collection = collect_pubmed_corpus(
    pubmed,
    PATHOGENS,
    OUTPUT_DIR,
    search_bundles=SEARCH_BUNDLES,
    max_results_per_query=PUBMED_MAX_RESULTS_PER_QUERY,
    search_page_size=PUBMED_SEARCH_PAGE_SIZE,
    fetch_batch_size=PUBMED_FETCH_BATCH_SIZE,
    resume=RESUME,
)
corpus = limit_debug_abstracts(collection.corpus, COMMON)
if DEBUG_MODE:
    corpus.to_parquet(OUTPUT_DIR / 'corpus_articles.parquet', index=False)
print('Corpus rows:', len(corpus))
print('Search failures:', collection.manifest['search_failures'])
print(corpus.groupby(['pathogen', 'fetch_status']).size())

## Refine retrieval with target-pathogen attribution

In [ ]:
if not OPENAI_API_KEY:
    raise RuntimeError('Set OPENAI_API_KEY before running the LLM refinement.')
llm = OpenAIChatCompleter(OPENAI_MODEL)
refinement_run = refine_pubmed_corpus(
    corpus,
    PATHOGENS,
    llm,
    OUTPUT_DIR,
    model=OPENAI_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    retries=LLM_RETRIES,
    max_llm_calls=LLM_MAX_CALLS,
    save_every=LLM_SAVE_EVERY,
    resume=RESUME,
    retry_failed=RETRY_FAILED_LLM_ROWS,
)
print('Candidate rows:', len(corpus))
print('Refinement decisions:', len(refinement_run.refinement))
print('Retained rows:', len(refinement_run.refined_corpus))
print('Rejected/needs review rows:', max(0, len(refinement_run.refinement) - len(refinement_run.refined_corpus) - len(refinement_run.failures)))
print('Retryable failures:', len(refinement_run.failures))

## Inspect refinement quality and provenance

Only rows with accepted target-pathogen attribution are written to
`refined_corpus_articles.parquet`. Rejected, ambiguous, and failed rows
remain in the refinement checkpoint for audit and retry; they are not
silently treated as biological negatives.

In [ ]:
import pandas as pd

# Always reconstruct the displayed state from checkpoints so this cell
# also works after manually interrupting the refinement cell.
corpus_checkpoint = OUTPUT_DIR / 'corpus_articles.parquet'
if corpus_checkpoint.exists():
    corpus = pd.read_parquet(corpus_checkpoint)
elif 'corpus' not in globals():
    raise FileNotFoundError(f'No corpus checkpoint found at {corpus_checkpoint}.')
refinement_run = load_refinement_run(corpus, OUTPUT_DIR)

print('Output directory:', OUTPUT_DIR)
print('Files:', sorted(path.name for path in OUTPUT_DIR.glob('*') if path.is_file()))
display(refinement_run.refinement[[
    'pathogen', 'pmid', 'refinement_keep', 'accepted',
    'target_pathogen_supported', 'evidence_relevant', 'confidence',
    'review_required', 'rationale'
]])